# S4 · AndinaLog 03B · Notebook 2 · Tratamiento controlado de Inventory Tracking

La salida `andinalog_inventory_tracking_silver.csv` contiene solo movimientos utilizables. El archivo tratado completo conserva las doce columnas Bronze, los valores preparados, las decisiones y la cuarentena para auditoría.

Este notebook trabaja con el Bronze oficial `andinalog_inventory_tracking.csv` y el diagnóstico v1 del notebook 1. Lee sus salidas y el `catalogo_reglas_tratamiento.csv` de esta carpeta. Solo aplica una regla cuando el catálogo indica `APROBADA`. La exclusión de copias exactamente idénticas conserva la aprobación documentada en el catálogo IoT del proyecto; las reglas nuevas para Inventory Tracking permanecen pendientes. No se inventan fechas, cantidades ni vencimientos, y no se fuerza ninguna fila a salir de cuarentena.

El catálogo es una decisión del proyecto, no una inferencia automática. Antes de cambiar una regla de `PENDIENTE` a `APROBADA`, documenta su evidencia y validación. El informe final se genera a partir de lo que **realmente ocurrió** en la ejecución.

## 1 · Configuración y entradas

En local, ejecuta desde cualquier carpeta dentro de `practicasNotebookColab`. En Colab, selecciona `drive` y ajusta la ruta de la carpeta que contiene `datasets/` y `proyecto-integrador/`. El notebook 2 lee las salidas del notebook 1 y el catálogo de reglas. Guarda los resultados en `proyecto-integrador/andinalog_inventory_tracking/notebook2/salidas/`.

In [1]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd

# ============================================================
# CONFIGURACIÓN
# ============================================================
ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO_REQUERIDA = "GIAD-M3-S4-INVENTORY-diagnostico-v1"
VERSION_TRATAMIENTO = "GIAD-M3-S4-INVENTORY-tratamiento-v1"
COLUMNAS_BRONZE = [
    "movimiento_id", "lote_id", "producto_id", "centro_distribucion",
    "fecha_ingreso", "fecha_salida", "fecha_vencimiento",
    "cantidad_ingreso", "cantidad_salida", "cantidad_merma",
    "dias_en_almacen", "costo_unitario_bob",
]

# ============================================================
# DETECCIÓN DE LA RAÍZ DEL PROYECTO EN LOCAL
# ============================================================
def raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if ((carpeta / "datasets" / "AndinaLog_03B_Bronce").is_dir()
                and (carpeta / "proyecto-integrador").is_dir()):
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto. Ejecuta dentro de practicasNotebookColab.")

# ============================================================
# CONFIGURACIÓN AUTOMÁTICA DE RUTAS
# ============================================================
def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'auto', 'local' o 'drive'")
    print(f"Entorno detectado: {entorno}")
    print(f"Raíz del proyecto: {raiz}")
    caso = raiz / "proyecto-integrador" / "andinalog_inventory_tracking"
    return {
        "bronze": raiz / "datasets" / "AndinaLog_03B_Bronce" / "andinalog_inventory_tracking.csv",
        "principal": caso / "notebook1" / "salidas" / "andinalog_inventory_tracking_diagnosticado.csv",
        "problemas": caso / "notebook1" / "salidas" / "andinalog_inventory_tracking_problemas.csv",
        "reporte1": caso / "notebook1" / "salidas" / "andinalog_inventory_tracking_reporte_calidad.csv",
        "catalogo": caso / "notebook2" / "catalogo_reglas_tratamiento.csv",
        "salidas": caso / "notebook2" / "salidas",
    }

# ============================================================
# OBTENER RUTAS Y VERIFICAR ENTRADAS
# ============================================================
rutas = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
for nombre in ["bronze", "principal", "problemas", "reporte1", "catalogo"]:
    if not rutas[nombre].is_file():
        raise FileNotFoundError(f"Falta {nombre}: {rutas[nombre]}")
print("Entradas verificadas.")
print("Directorio de salidas:", rutas["salidas"])

Entorno detectado: local
Raíz del proyecto: c:\Users\remrodri\Github\practicasNotebookColab
Entradas verificadas.
Directorio de salidas: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook2\salidas


## 2 · Lectura y validación del contrato

El número `fila_bronze` vincula el archivo principal con el detalle de problemas. La huella del reporte 1 debe corresponder al CSV Bronze actual; si el origen cambió, se debe volver a ejecutar el notebook 1 antes de tratar. El catálogo debe contener una regla por identificador, con evidencia para cada regla aprobada.

In [2]:
def leer_entradas(rutas):
    principal = pd.read_csv(rutas["principal"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    problemas = pd.read_csv(rutas["problemas"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    reporte1 = pd.read_csv(rutas["reporte1"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    catalogo = pd.read_csv(rutas["catalogo"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return principal, problemas, reporte1, catalogo

def validar_entradas(principal, problemas, reporte1, catalogo, ruta_bronze):
    requeridas = ["fila_bronze", *COLUMNAS_BRONZE, "columnas_con_problemas", "en_cuarentena"]
    if list(principal.columns) != requeridas:
        raise ValueError("El principal no coincide con las columnas de Notebook 1 Inventory Tracking")
    if not {"fila_bronze", "columna_afectada", "codigo_error", "valor_original", "version_diagnostico"}.issubset(problemas.columns):
        raise ValueError("El detalle de problemas no cumple su contrato")
    columnas_catalogo = ["regla_id", "columna_afectada", "codigo_error_o_unidad", "tratamiento_propuesto", "estado", "evidencia_acuerdo", "validacion_requerida"]
    if list(catalogo.columns) != columnas_catalogo:
        raise ValueError("El catálogo no cumple su contrato")
    if principal["fila_bronze"].duplicated().any() or catalogo["regla_id"].duplicated().any():
        raise ValueError("Hay identificadores duplicados")
    if not problemas["fila_bronze"].isin(principal["fila_bronze"]).all():
        raise ValueError("Hay problemas sin fila en el principal")
    if not catalogo["estado"].isin(["APROBADA", "PENDIENTE"]).all():
        raise ValueError("Estado de regla desconocido")
    if (catalogo["estado"].eq("APROBADA") & catalogo["evidencia_acuerdo"].str.strip().eq("")).any():
        raise ValueError("Hay reglas aprobadas sin evidencia del acuerdo")
    if (catalogo["estado"].eq("APROBADA") & catalogo["validacion_requerida"].str.strip().eq("")).any():
        raise ValueError("Hay reglas aprobadas sin validación requerida")
    resumen = reporte1.set_index("metrica")["valor"]
    if resumen.loc["version_diagnostico"] != VERSION_DIAGNOSTICO_REQUERIDA:
        raise ValueError("Se requiere el diagnóstico v1 de Inventory Tracking; vuelve a ejecutar Notebook 1")
    hash_actual = hashlib.sha256(ruta_bronze.read_bytes()).hexdigest()
    if resumen.loc["sha256_bronze"] != hash_actual:
        raise ValueError("El Bronze ya no coincide con el diagnóstico; ejecuta Notebook 1")
    if len(principal) != int(resumen.loc["filas_bronze"]):
        raise ValueError("El principal no coincide con el conteo Bronze")
    if len(problemas) and not problemas["version_diagnostico"].eq(VERSION_DIAGNOSTICO_REQUERIDA).all():
        raise ValueError("El detalle de problemas tiene otra versión de diagnóstico")
    bronze = pd.read_csv(ruta_bronze, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    pd.testing.assert_frame_equal(principal[COLUMNAS_BRONZE].reset_index(drop=True), bronze[COLUMNAS_BRONZE])
    return hash_actual

df_entrada, problemas_entrada, reporte1, catalogo = leer_entradas(rutas)
HASH_BRONZE = validar_entradas(df_entrada, problemas_entrada, reporte1, catalogo, rutas["bronze"])
print(f"Filas: {len(df_entrada):,}; problemas iniciales: {len(problemas_entrada):,}")
display(catalogo)

Filas: 6,040; problemas iniciales: 134


,regla_id,columna_afectada,codigo_error_o_unidad,tratamiento_propuesto,estado,evidencia_acuerdo,validacion_requerida
0,DUPLICADO_IDENTICO,movimiento_id,DUPLICADO_IDENTICO,Elegir primera fila canónica y excluir copia e...,APROBADA,Regla de copias idénticas aprobada en el catál...,Las doce columnas Bronze deben ser idénticas; ...
1,DUPLICADO_EQUIVALENTE_FECHA,movimiento_id,CLAVE_EN_CONFLICTO,Elegir primera fila si la única diferencia des...,PENDIENTE,,Requiere FECHA_DDMM_A_ISO aprobada; dos filas ...
2,DUPLICADO_CONFLICTIVO,movimiento_id,CLAVE_EN_CONFLICTO,Conservar ambas filas en cuarentena,PENDIENTE,,Fuente autoritativa para decidir si hay dos mo...
3,FECHA_DDMM_A_ISO,fecha_ingreso|fecha_salida|fecha_vencimiento,FORMATO_FECHA_DISTINTO,Interpretar fecha válida DD/MM/YYYY y preparar...,PENDIENTE,,Fecha válida en formato DD/MM/YYYY; revisar dí...
4,PRODUCTO_MAYUSCULAS,producto_id,FORMATO_INVALIDO,Preparar identificador en mayúsculas solo si e...,PENDIENTE,,Confirmar equivalencia de producto con catálog...
5,FECHA_VENCIMIENTO_FALTANTE,fecha_vencimiento,FALTANTE,Conservar en cuarentena sin imputación automática,PENDIENTE,,Vencimiento verificable en fuente del lote
6,FECHA_IMPOSIBLE,fecha_ingreso|fecha_salida|fecha_vencimiento,FECHA_INVALIDA,Conservar en cuarentena sin inventar fecha,PENDIENTE,,Fecha original recuperada de fuente autorizada
7,CANTIDAD_INGRESO_NO_NUMERICA,cantidad_ingreso,NO_NUMERICA,Conservar en cuarentena; no traducir 'cien' au...,PENDIENTE,,Cantidad numérica confirmada en documento de i...
8,MERMA_NEGATIVA,cantidad_merma,NEGATIVA,Conservar en cuarentena; no cambiar signo auto...,PENDIENTE,,Merma real confirmada por registro operativo
9,COHERENCIA_TEMPORAL,fecha_salida|fecha_vencimiento|dias_en_almacen,ANTERIOR_INGRESO|NO_COINCIDE_FECHAS,Conservar en cuarentena hasta aclarar secuenci...,PENDIENTE,,Fechas y duración reconciliadas con fuente aut...


## 3 · Aplicación de reglas aprobadas

Los doce valores Bronze se conservan. Las fechas normalizadas y el identificador de producto preparado se guardan en columnas nuevas solo cuando sus reglas se aprueban y se validan fila por fila. La copia exacta posterior se excluye de Silver según la regla heredada del catálogo IoT. Un conflicto de `movimiento_id` solo se resuelve si las dos filas quedan idénticas tras una normalización de fecha aprobada y existe una aprobación separada para elegir la canónica. Fechas imposibles, vencimientos faltantes y cantidades anómalas permanecen pendientes sin datos de respaldo.

In [3]:
def regla_aprobada(catalogo, regla_id):
    fila = catalogo.loc[catalogo["regla_id"].eq(regla_id)]
    if len(fila) != 1:
        raise ValueError(f"Falta regla única: {regla_id}")
    if fila.iloc[0]["estado"] == "APROBADA":
        if not str(fila.iloc[0]["evidencia_acuerdo"]).strip():
            raise ValueError(f"La regla {regla_id} figura aprobada sin evidencia")
        return True
    return False

def preparar_valores(principal, catalogo):
    df = principal.copy(deep=True)
    for col in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]:
        df[col + "_preparada"] = df[col]
    df["producto_id_preparado"] = df["producto_id"]
    df["tratamientos_preparacion"] = ""
    if regla_aprobada(catalogo, "FECHA_DDMM_A_ISO"):
        for col in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]:
            original = df[col].str.strip()
            formato = original.str.fullmatch(r"\d{2}/\d{2}/\d{4}").fillna(False)
            fecha = pd.to_datetime(original.where(formato), format="%d/%m/%Y", errors="coerce")
            mascara = formato & fecha.notna()
            df.loc[mascara, col + "_preparada"] = fecha.loc[mascara].dt.strftime("%Y-%m-%d")
    if regla_aprobada(catalogo, "PRODUCTO_MAYUSCULAS"):
        candidato = df["producto_id"].str.strip().str.upper()
        validos = set(df.loc[df["producto_id"].str.fullmatch(r"PROD-\d{3}").fillna(False), "producto_id"])
        mascara = (df["producto_id"].ne(candidato) & candidato.str.fullmatch(r"PROD-\d{3}").fillna(False)
                   & candidato.isin(validos))
        df.loc[mascara, "producto_id_preparado"] = candidato.loc[mascara]
    return df

def analizar_duplicados(principal, catalogo):
    registros = []
    exacta_aprobada = regla_aprobada(catalogo, "DUPLICADO_IDENTICO")
    fecha_aprobada = regla_aprobada(catalogo, "FECHA_DDMM_A_ISO")
    equivalente_aprobada = regla_aprobada(catalogo, "DUPLICADO_EQUIVALENTE_FECHA")
    for clave, grupo in principal.groupby(principal["movimiento_id"].str.strip(), sort=False):
        if not clave or len(grupo) == 1:
            continue
        grupo = grupo.sort_values("fila_bronze", key=lambda s: s.astype(int))
        exacto = grupo[COLUMNAS_BRONZE].drop_duplicates().shape[0] == 1
        comparadas = grupo[COLUMNAS_BRONZE].copy()
        for col in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]:
            comparadas[col] = grupo[col + "_preparada"]
        equivalente_fecha = (not exacto and fecha_aprobada and len(grupo) == 2
                             and comparadas.drop_duplicates().shape[0] == 1)
        tipo = "IDENTICO" if exacto else ("EQUIVALENTE_FECHA" if equivalente_fecha else "CONFLICTO_O_AMBIGUO")
        autorizada = (tipo == "IDENTICO" and exacta_aprobada) or (tipo == "EQUIVALENTE_FECHA" and equivalente_aprobada)
        canonica = grupo.iloc[0]["fila_bronze"] if autorizada else ""
        evidencia = ("Las doce columnas Bronze son idénticas" if exacto else
                     "Las doce columnas coinciden tras normalizar la fecha aprobada" if equivalente_fecha else
                     "Hay valores contradictorios o falta aprobación para interpretar la diferencia")
        for _, fila in grupo.iterrows():
            decision = ("CANONICA" if fila["fila_bronze"] == canonica else "COPIA_EXCLUIDA") if autorizada else "PENDIENTE"
            registros.append({"fila_bronze": fila["fila_bronze"], "movimiento_id": clave,
                              "tipo_duplicado": tipo, "decision_duplicado": decision,
                              "fila_canonica": canonica, "justificacion_duplicado": evidencia})
    return pd.DataFrame(registros, columns=["fila_bronze", "movimiento_id", "tipo_duplicado", "decision_duplicado", "fila_canonica", "justificacion_duplicado"])

def evaluar_problemas(principal, problemas, catalogo, decisiones):
    acciones = problemas.copy(deep=True)
    acciones["estado_tratamiento"] = "PENDIENTE"
    acciones["tratamiento_aplicado"] = "NINGUNO"
    acciones["detalle_resultado"] = "No hay regla aprobada y validada para resolver este problema"
    preparadas = principal.set_index("fila_bronze")
    if regla_aprobada(catalogo, "FECHA_DDMM_A_ISO"):
        formato = acciones["codigo_error"].eq("FORMATO_FECHA_DISTINTO")
        for col in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]:
            m = formato & acciones["columna_afectada"].eq(col)
            nueva = acciones.loc[m, "fila_bronze"].map(preparadas[col + "_preparada"])
            valida = nueva.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False).to_numpy()
            indices = acciones.loc[m].index[valida]
            acciones.loc[indices, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
                "RESUELTO", "FECHA_DDMM_A_ISO", "Fecha válida interpretada como DD/MM/YYYY y preparada como YYYY-MM-DD"]
    if regla_aprobada(catalogo, "PRODUCTO_MAYUSCULAS"):
        m = acciones["columna_afectada"].eq("producto_id") & acciones["codigo_error"].eq("FORMATO_INVALIDO")
        nueva = acciones.loc[m, "fila_bronze"].map(preparadas["producto_id_preparado"])
        original = acciones.loc[m, "fila_bronze"].map(preparadas["producto_id"])
        indices = acciones.loc[m].index[nueva.ne(original).to_numpy()]
        acciones.loc[indices, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
            "RESUELTO", "PRODUCTO_MAYUSCULAS", "Identificador preparado y validado contra un identificador canónico del lote"]
    if len(decisiones):
        por_fila = decisiones.set_index("fila_bronze")
        es_duplicado = acciones["codigo_error"].isin(["DUPLICADO_IDENTICO", "CLAVE_EN_CONFLICTO"])
        decision = acciones["fila_bronze"].map(por_fila["decision_duplicado"])
        canonica = es_duplicado & decision.eq("CANONICA")
        excluida = es_duplicado & decision.eq("COPIA_EXCLUIDA")
        acciones.loc[canonica, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
            "RESUELTO", "SELECCION_CANONICA", "Movimiento canónico elegido según regla aprobada"]
        acciones.loc[excluida, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
            "EXCLUIDO_COMO_COPIA", "COPIA_EXCLUIDA", "Copia conservada para auditoría; fuera de Silver"]
        # El diagnóstico marca solo la copia exacta, no la primera fila canónica.
        # Si una decisión excluye otra fila sin motivo de duplicado, se registra aquí.
        con_motivo = set(acciones.loc[es_duplicado, "fila_bronze"])
        nuevas = decisiones.loc[decisiones["decision_duplicado"].eq("COPIA_EXCLUIDA")
                                & ~decisiones["fila_bronze"].isin(con_motivo)]
        if len(nuevas):
            extra = pd.DataFrame({"fila_bronze": nuevas["fila_bronze"],
                                  "columna_afectada": "movimiento_id", "codigo_error": "COPIA_EXCLUIDA",
                                  "valor_original": nuevas["movimiento_id"],
                                  "version_diagnostico": VERSION_DIAGNOSTICO_REQUERIDA,
                                  "estado_tratamiento": "EXCLUIDO_COMO_COPIA",
                                  "tratamiento_aplicado": "COPIA_EXCLUIDA",
                                  "detalle_resultado": "Movimiento no canónico conservado para auditoría"})
            acciones = pd.concat([acciones, extra], ignore_index=True)
    return acciones.sort_values(["fila_bronze", "columna_afectada", "codigo_error"],
                                key=lambda s: s.astype(int) if s.name == "fila_bronze" else s,
                                kind="stable").reset_index(drop=True)

df_trabajo = preparar_valores(df_entrada, catalogo)
decisiones_duplicado = analizar_duplicados(df_trabajo, catalogo)
acciones = evaluar_problemas(df_trabajo, problemas_entrada, catalogo, decisiones_duplicado)
print(decisiones_duplicado.groupby(["tipo_duplicado", "decision_duplicado"]).size().to_string())
print(acciones["estado_tratamiento"].value_counts().to_string())

tipo_duplicado       decision_duplicado
CONFLICTO_O_AMBIGUO  PENDIENTE              2
IDENTICO             CANONICA              39
                     COPIA_EXCLUIDA        39
estado_tratamiento
PENDIENTE              95
EXCLUIDO_COMO_COPIA    39


## 4 · Estado final y conciliación

Solo `RESUELTO` deja de ser motivo pendiente. `EXCLUIDO_COMO_COPIA` conserva la fila en cuarentena para impedir que duplique un movimiento, aunque su tratamiento queda registrado. Una fila sale de cuarentena únicamente cuando no le queda ningún problema pendiente o excluido. Silver usa los valores preparados únicamente de las filas liberadas.

In [4]:
def construir_estado_final(principal, acciones, decisiones):
    df = principal.copy(deep=True)
    por_fila = decisiones.set_index("fila_bronze")
    for campo in ["tipo_duplicado", "decision_duplicado", "fila_canonica", "justificacion_duplicado"]:
        df[campo] = df["fila_bronze"].map(por_fila[campo]).fillna("") if len(decisiones) else ""
    df["en_cuarentena_inicial"] = df["en_cuarentena"].str.lower().eq("true")
    df["columnas_problema_iniciales"] = df["columnas_con_problemas"]
    pendientes = acciones.loc[~acciones["estado_tratamiento"].eq("RESUELTO")].copy()
    pendientes["motivo"] = pendientes["columna_afectada"] + ":" + pendientes["codigo_error"]
    motivos = pendientes.groupby("fila_bronze")["motivo"].agg(lambda v: "|".join(dict.fromkeys(v)))
    df["motivos_finales"] = df["fila_bronze"].map(motivos).fillna("")
    df["en_cuarentena_final"] = df["motivos_finales"].ne("")
    tratamientos = acciones.loc[acciones["tratamiento_aplicado"].ne("NINGUNO")].groupby("fila_bronze")["tratamiento_aplicado"].agg(lambda v: "|".join(dict.fromkeys(v)))
    df["tratamientos_aplicados"] = df["fila_bronze"].map(tratamientos).fillna("")
    df["version_tratamiento"] = VERSION_TRATAMIENTO
    return df, pendientes

df_final, problemas_pendientes = construir_estado_final(df_trabajo, acciones, decisiones_duplicado)
df_cuarentena_final = df_final.loc[df_final["en_cuarentena_final"]].copy()
df_silver = df_final.loc[~df_final["en_cuarentena_final"], [
    "movimiento_id", "lote_id", "producto_id_preparado", "centro_distribucion",
    "fecha_ingreso_preparada", "fecha_salida_preparada", "fecha_vencimiento_preparada",
    "cantidad_ingreso", "cantidad_salida", "cantidad_merma", "dias_en_almacen", "costo_unitario_bob",
]].copy()
df_silver.columns = COLUMNAS_BRONZE

pd.testing.assert_frame_equal(df_final[df_entrada.columns], df_entrada)
assert len(df_final) == len(df_entrada)
assert len(df_cuarentena_final) == int(df_final["en_cuarentena_final"].sum())
assert len(df_silver) == int((~df_final["en_cuarentena_final"]).sum())
assert list(df_silver.columns) == COLUMNAS_BRONZE
assert not df_silver["movimiento_id"].duplicated().any()
assert set(problemas_pendientes["fila_bronze"]) == set(df_cuarentena_final["fila_bronze"])
assert df_final.loc[df_final["decision_duplicado"].eq("COPIA_EXCLUIDA"), "en_cuarentena_final"].all()
assert not df_final.loc[~df_final["en_cuarentena_final"], "movimiento_id"].duplicated().any()
if len(decisiones_duplicado):
    seleccion = decisiones_duplicado.groupby("movimiento_id")["decision_duplicado"].value_counts().unstack(fill_value=0)
    aprobadas = seleccion.loc[seleccion.get("PENDIENTE", pd.Series(0, index=seleccion.index)).eq(0)]
    assert aprobadas.get("CANONICA", pd.Series(0, index=aprobadas.index)).eq(1).all()
print("Filas iniciales en cuarentena:", int(df_final["en_cuarentena_inicial"].sum()))
print("Filas finales en cuarentena:", int(df_final["en_cuarentena_final"].sum()))
print("Filas liberadas:", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum()))

Filas iniciales en cuarentena: 133
Filas finales en cuarentena: 133
Filas liberadas: 0


## 5 · Informe de lo aplicado y exportación

El reporte registra las reglas aprobadas, los problemas resueltos, los pendientes, las copias excluidas y el número real de filas liberadas. No atribuye una liberación a una normalización si la fila mantiene otro motivo.

In [5]:
def crear_reporte(df_final, acciones, decisiones, catalogo, huella):
    base = [
        ("sha256_bronze", huella),
        ("version_tratamiento", VERSION_TRATAMIENTO),
        ("filas_totales", len(df_final)),
        ("filas_cuarentena_inicial", int(df_final["en_cuarentena_inicial"].sum())),
        ("filas_cuarentena_final", int(df_final["en_cuarentena_final"].sum())),
        ("filas_liberadas", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum())),
        ("filas_silver", int((~df_final["en_cuarentena_final"]).sum())),
        ("problemas_resueltos", int(acciones["estado_tratamiento"].eq("RESUELTO").sum())),
        ("copias_excluidas", int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())),
        ("problemas_pendientes", int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())),
        ("fechas_normalizadas", int(acciones["tratamiento_aplicado"].eq("FECHA_DDMM_A_ISO").sum())),
        ("productos_normalizados", int(acciones["tratamiento_aplicado"].eq("PRODUCTO_MAYUSCULAS").sum())),
    ]
    canonicas = decisiones.loc[decisiones["decision_duplicado"].eq("CANONICA")]
    base += [("pares_duplicados_" + tipo.lower(), int(canonicas["tipo_duplicado"].eq(tipo).sum()))
             for tipo in ["IDENTICO", "EQUIVALENTE_FECHA"]]
    base += [("duplicados_sin_decision", int(decisiones["decision_duplicado"].eq("PENDIENTE").sum()))]
    base += [("regla_" + r["regla_id"], r["estado"]) for _, r in catalogo.iterrows()]
    return pd.DataFrame(base, columns=["metrica", "valor"])

def exportar(directorio, tablas, bronze, huella):
    if hashlib.sha256(bronze.read_bytes()).hexdigest() != huella:
        raise RuntimeError("El Bronze cambió; se cancela la exportación")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_inventory2_", dir=directorio,
                                             encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

reporte2 = crear_reporte(df_final, acciones, decisiones_duplicado, catalogo, HASH_BRONZE)
tablas = {
    "andinalog_inventory_tracking_tratado.csv": df_final,
    "andinalog_inventory_tracking_silver.csv": df_silver,
    "andinalog_inventory_tracking_acciones.csv": acciones,
    "andinalog_inventory_tracking_decisiones_duplicados.csv": decisiones_duplicado,
    "andinalog_inventory_tracking_cuarentena_final.csv": df_cuarentena_final,
    "andinalog_inventory_tracking_reporte_tratamiento.csv": reporte2,
}
for ruta in exportar(rutas["salidas"], tablas, rutas["bronze"], HASH_BRONZE):
    print(ruta)
display(reporte2)

c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_tratado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_silver.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_acciones.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_decisiones_duplicados.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_cuarentena_final.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_reporte_tratamiento.csv


,metrica,valor
0,sha256_bronze,4142c8736512c78c553bdd78dd49b6fe304039f7deec8c...
1,version_tratamiento,GIAD-M3-S4-INVENTORY-tratamiento-v1
2,filas_totales,6040
3,filas_cuarentena_inicial,133
4,filas_cuarentena_final,133
5,filas_liberadas,0
6,filas_silver,5907
7,problemas_resueltos,0
8,copias_excluidas,39
9,problemas_pendientes,95


In [6]:
def crear_informe_md(df_final, acciones, decisiones, catalogo, huella):
    inicial = int(df_final["en_cuarentena_inicial"].sum())
    final = int(df_final["en_cuarentena_final"].sum())
    resueltos = int(acciones["estado_tratamiento"].eq("RESUELTO").sum())
    pendientes = int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())
    excluidos = int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())
    canonicas = decisiones.loc[decisiones["decision_duplicado"].eq("CANONICA")]
    exactos = int(canonicas["tipo_duplicado"].eq("IDENTICO").sum())
    equivalentes = int(canonicas["tipo_duplicado"].eq("EQUIVALENTE_FECHA").sum())
    fechas = int(acciones["tratamiento_aplicado"].eq("FECHA_DDMM_A_ISO").sum())
    productos = int(acciones["tratamiento_aplicado"].eq("PRODUCTO_MAYUSCULAS").sum())
    conteos = acciones.loc[~acciones["estado_tratamiento"].eq("RESUELTO")].groupby(["columna_afectada", "codigo_error"]).size()
    lineas = [
        "# Informe de tratamiento de Inventory Tracking", "",
        f"**Fuente Bronze SHA-256:** `{huella}`",
        f"**Versión:** `{VERSION_TRATAMIENTO}`", "",
        "## Resultado del lote", "",
        f"Se conservaron las {len(df_final):,} filas Bronze. La cuarentena pasó de {inicial:,} a {final:,} filas; {inicial-final:,} salieron después de resolver todos sus motivos. Silver contiene {len(df_final)-final:,} movimientos. Se registraron {resueltos:,} problemas resueltos, {pendientes:,} pendientes y {excluidos:,} copias excluidas.", "",
        "## Tratamientos aplicados", "",
        f"- Se prepararon {fechas:,} fechas válidas `DD/MM/YYYY` como `YYYY-MM-DD`, conservando el valor Bronze.",
        f"- Se prepararon {productos:,} identificadores de producto en mayúsculas cuando coincidían con un identificador canónico presente en el lote.",
        "- No se imputaron fechas de vencimiento ni cantidades; no se corrigieron fechas imposibles ni mermas negativas por suposición.",
        "- Una salida más merma menor que el ingreso se admite como posible stock remanente; solo el exceso sería inconsistente.", "",
        "## Duplicados y selección canónica", "",
        f"Se eligieron {exactos:,} movimientos canónicos de pares exactamente idénticos y {equivalentes:,} de pares equivalentes tras una normalización de fecha aprobada. Se conservaron {excluidos:,} copias excluidas para auditoría.",
        "Los conflictos sin evidencia suficiente siguen pendientes; cada decisión consta en `andinalog_inventory_tracking_decisiones_duplicados.csv`.", "",
        "## Motivos que permanecen en cuarentena", "",
        "| Columna | Código | Motivos finales |", "|---|---|---:|",
    ]
    lineas += [f"| {col} | {codigo} | {int(total)} |" for (col, codigo), total in conteos.items()]
    lineas += ["", "Una fila puede tener varios motivos. El CSV de acciones detalla cada intento y el archivo tratado conserva el estado inicial y final.", "", "## Reglas y límites de esta ejecución", ""]
    lineas += [f"- `{r['regla_id']}`: **{r['estado']}**. Tratamiento: {r['tratamiento_propuesto']}. Validación: {r['validacion_requerida']}. Acuerdo: {r['evidencia_acuerdo'] or 'pendiente.'}"
               for _, r in catalogo.iterrows()]
    lineas += ["", "Una fila permaneció en cuarentena cuando no había una regla aprobada, faltaba evidencia o quedaba otro motivo sin resolver. No se forzó ninguna liberación.", ""]
    return "\n".join(lineas)

informe_md = crear_informe_md(df_final, acciones, decisiones_duplicado, catalogo, HASH_BRONZE)
ruta_informe = rutas["salidas"] / "Informe_S4_02_Tratamiento.md"
ruta_informe.write_text(informe_md, encoding="utf-8")
print(ruta_informe)

c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook2\salidas\Informe_S4_02_Tratamiento.md


## 6 · Interpretación de esta ejecución

Revisa el reporte y el archivo de acciones antes de afirmar que una fila salió de cuarentena. Las reglas `PENDIENTE` necesitan evidencia o acuerdo; por eso no se inventa un vencimiento, una fecha de salida o una cantidad de ingreso. El informe debe describir los tratamientos realmente ejecutados y los casos que permanecen en cuarentena.